<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/05-bias-variance.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 5 — Bias, Variance and Regularisation

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/bias-variance-and-regularisation/).

Back in the one-feature delivery world, where we know the truth and can split a model's error into the three pieces it is made of.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def bicycle(km):
    return 10 + 3.2 * km + 0.15 * km**2

def make_orders(n, noise=4.0, seed=0, lo=0.5, hi=8.0):
    rng = np.random.default_rng(seed)
    x = rng.uniform(lo, hi, n)
    return x, bicycle(x) + rng.normal(0, noise, n)

def poly_features(x, degree):
    """Columns 1, x, x², ... up to x^degree, with x scaled to [-1, 1] first so the powers stay sane."""
    z = (x - 4.25) / 3.75
    return np.column_stack([z**d for d in range(degree + 1)])

def fit_poly(x, y, degree):
    return np.linalg.lstsq(poly_features(x, degree), y, rcond=None)[0]

def predict_poly(w, x):
    return poly_features(x, len(w) - 1) @ w

grid = np.linspace(0.5, 8.0, 50)       # the places where we will measure every model
truth = bicycle(grid)

## 1. The same family, fitted to many different months

In [ ]:
def many_fits(degree, n=30, trials=200, noise=4.0):
    """Fit the same polynomial to `trials` different samples of n orders. Return all the prediction curves."""
    curves = np.empty((trials, len(grid)))
    for t in range(trials):
        x, y = make_orders(n, noise=noise, seed=1000 + t)
        curves[t] = predict_poly(fit_poly(x, y, degree), grid)
    return curves

noise = 4.0
print(f'{"degree":>6} {"bias²":>7} {"variance":>9} {"noise²":>7} {"total":>7}')
for degree in [0, 1, 2, 3, 5, 8, 12]:
    curves = many_fits(degree)
    mean_curve = curves.mean(axis=0)
    bias2 = np.mean((mean_curve - truth) ** 2)
    variance = np.mean(curves.var(axis=0))
    print(f'{degree:>6} {bias2:>7.2f} {variance:>9.2f} {noise**2:>7.2f} {bias2 + variance + noise**2:>7.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for ax, degree in zip(axes, [0, 2, 12]):
    curves = many_fits(degree, trials=30)
    for c in curves:
        ax.plot(grid, c, color='steelblue', alpha=0.25, lw=1)
    ax.plot(grid, truth, color='black', lw=2, label='truth')
    ax.set_ylim(0, 70); ax.set_title(f'degree {degree}: 30 months, 30 fits'); ax.set_xlabel('km')
axes[0].set_ylabel('minutes'); axes[0].legend()
plt.tight_layout(); plt.show()

## 2. Where the error comes from, checked against a test set

In [ ]:
x_test, y_test = make_orders(2000, seed=99)
print(f'{"degree":>6} {"train MSE":>10} {"test MSE":>9}')
for degree in [0, 1, 2, 3, 5, 8, 12]:
    x, y = make_orders(30, seed=1000)
    w = fit_poly(x, y, degree)
    train = np.mean((predict_poly(w, x) - y) ** 2)
    test = np.mean((predict_poly(w, x_test) - y_test) ** 2)
    print(f'{degree:>6} {train:>10.2f} {test:>9.2f}')

## 3. More data moves the balance

In [ ]:
print(f'{"n":>5} {"degree 1":>9} {"degree 2":>9} {"degree 8":>9}')
for n in [15, 30, 100, 1000]:
    row = []
    for degree in [1, 2, 8]:
        curves = many_fits(degree, n=n, trials=100)
        row.append(np.mean((curves - truth) ** 2))      # bias² + variance, measured directly
    print(f'{n:>5} ' + ' '.join(f'{r:>9.2f}' for r in row))

## 4. Learning curves: which problem do I have?

In [ ]:
def learning_curve(degree, sizes):
    rows = []
    for n in sizes:
        tr, te = [], []
        for t in range(20):
            x, y = make_orders(n, seed=5000 + t)
            w = fit_poly(x, y, degree)
            tr.append(np.mean((predict_poly(w, x) - y) ** 2))
            te.append(np.mean((predict_poly(w, x_test) - y_test) ** 2))
        rows.append((n, np.mean(tr), np.mean(te)))
    return rows

sizes = [20, 40, 80, 160, 320]
for degree in [0, 8]:
    print(f'degree {degree}:  {"n":>4} {"train":>7} {"test":>7}')
    for n, tr, te in learning_curve(degree, sizes):
        print(f'           {n:>4} {tr:>7.2f} {te:>7.2f}')

## 5. Ridge: pay for every unit of weight

In [ ]:
def fit_ridge(x, y, degree, alpha):
    Z = poly_features(x, degree)
    penalty = alpha * np.eye(Z.shape[1])
    penalty[0, 0] = 0                                 # never penalise the intercept
    return np.linalg.solve(Z.T @ Z + penalty, Z.T @ y)

x, y = make_orders(30, seed=1000)
print(f'{"alpha":>8} {"test MSE":>9}  {"size of weights":>15}')
for alpha in [0, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10]:
    w = fit_ridge(x, y, 12, alpha)
    test = np.mean((predict_poly(w, x_test) - y_test) ** 2)
    print(f'{alpha:>8} {test:>9.2f}  {np.abs(w[1:]).sum():>15.1f}')

In [ ]:
print('degree-12 weights, alpha = 0   :', np.round(fit_ridge(x, y, 12, 0)[1:6], 1), '...')
print('degree-12 weights, alpha = 0.01:', np.round(fit_ridge(x, y, 12, 0.01)[1:6], 1), '...')

## 6. Ridge through the bias-variance lens

In [ ]:
def many_ridge(degree, alpha, n=30, trials=200):
    curves = np.empty((trials, len(grid)))
    for t in range(trials):
        xx, yy = make_orders(n, seed=1000 + t)
        curves[t] = predict_poly(fit_ridge(xx, yy, degree, alpha), grid)
    return curves

print(f'{"alpha":>8} {"bias²":>7} {"variance":>9} {"total":>7}')
for alpha in [0, 1e-3, 1e-2, 1e-1, 1]:
    curves = many_ridge(12, alpha)
    bias2 = np.mean((curves.mean(axis=0) - truth) ** 2)
    variance = np.mean(curves.var(axis=0))
    print(f'{alpha:>8} {bias2:>7.2f} {variance:>9.2f} {bias2 + variance + 16:>7.2f}')

## 7. Lasso: pay for every weight that is not zero

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline

Z = poly_features(x, 12)[:, 1:]                     # drop the constant column; the models add their own intercept
Z_test = poly_features(x_test, 12)[:, 1:]
for name, model in [('ridge  alpha=0.1', Ridge(alpha=0.1)),
                    ('lasso  alpha=0.01', Lasso(alpha=0.01, max_iter=50000)),
                    ('lasso  alpha=0.1', Lasso(alpha=0.1, max_iter=50000))]:
    m = make_pipeline(StandardScaler(), model).fit(Z, y)
    coef = m[-1].coef_
    test = np.mean((m.predict(Z_test) - y_test) ** 2)
    print(f'{name:<18} test MSE {test:6.2f}   weights exactly zero: {np.sum(coef == 0):>2} of 12')

## 8. Collinear features, revisited

In [ ]:
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(1)
km_m = rng.uniform(0.5, 8, 200)
items_m = rng.integers(1, 7, 200)
minutes_m = 8 + 4 * km_m + 1.5 * items_m + rng.normal(0, 4, 200)
miles = km_m / 1.609 + rng.normal(0, 0.02, 200)          # the same distance again, from a noisier odometer
F = np.column_stack([km_m, miles, items_m])
print(f'{"":<10} {"km":>7} {"miles":>7} {"items":>7}   (weights on standardised features)')
for name, model in [('plain', LinearRegression()), ('ridge', Ridge(alpha=1.0))]:
    m = make_pipeline(StandardScaler(), model).fit(F, minutes_m)
    print(f'{name:<10}', ' '.join(f'{c:>7.2f}' for c in m[-1].coef_))

## 9. Choosing alpha without touching the test set

In [ ]:
def cv_mse(x, y, degree, alpha, folds=5):
    """Mean squared error on held-out folds, averaged over the folds."""
    idx = np.arange(len(x))
    parts = np.array_split(idx, folds)
    errors = []
    for held in parts:
        train = np.setdiff1d(idx, held)
        w = fit_ridge(x[train], y[train], degree, alpha)
        errors.append(np.mean((predict_poly(w, x[held]) - y[held]) ** 2))
    return np.mean(errors)

print(f'{"alpha":>8} {"CV MSE":>8} {"test MSE":>9}')
best_alpha, best_cv = None, np.inf
for alpha in [1e-4, 1e-3, 1e-2, 1e-1, 1, 10]:
    cv = cv_mse(x, y, 12, alpha)
    test = np.mean((predict_poly(fit_ridge(x, y, 12, alpha), x_test) - y_test) ** 2)
    print(f'{alpha:>8} {cv:>8.2f} {test:>9.2f}')
    if cv < best_cv:
        best_alpha, best_cv = alpha, cv
print(f'cross-validation picks alpha = {best_alpha}, having never seen the test set')

## 10. The one line you would ship

In [ ]:
from sklearn.linear_model import RidgeCV

model = make_pipeline(PolynomialFeatures(degree=12, include_bias=False), StandardScaler(),
                      RidgeCV(alphas=np.logspace(-3, 3, 13)))      # leave-one-out, done efficiently
model.fit(x.reshape(-1, 1), y)
print(f'test MSE: {np.mean((model.predict(x_test.reshape(-1, 1)) - y_test) ** 2):.2f}   chosen alpha {model[-1].alpha_:.3g}')

## Try it yourself

1. Set `noise=1.0` in section 1. Which degree now has the lowest total, and why has it moved?
2. Add a thirteenth column that is pure noise to the lasso in section 7. Does it get a zero?
3. In section 9, change `cv=5` to `cv=2` and to `cv=15`. How stable is the chosen alpha across different `seed` values for the training data?